# Real Data Examples

In [ ]:
import importlib
import subprocess

def install_if_missing(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        subprocess.run(["pip", "install", package], check=True)

In [ ]:
import os
import sys
import torch
import time

from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import argparse
import torch.nn as nn
import torch.optim as optim
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy.spatial.distance import cdist, pdist
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle
import traceback
install_if_missing("pyreadr")
import pyreadr
from flowgem import sample_flowgem
from sklearn.impute import KNNImputer
import itertools

from scipy.optimize import minimize
from scipy.stats import multivariate_normal

install_if_missing("hyperimpute")
from hyperimpute.plugins.imputers import Imputers
import scipy

install_if_missing("pot")
import ot

base_path = "./"
sys.path.append(str(Path("MIRI-Imputation").resolve()))
from src.imputer_wrapper import impute_now

## Set parameters

In [ ]:
### SETTINGS ###

datasets = ["parkinsons", "allergens", "concrete", "windspeed", "forest", "housing", "stock", "pumadyn32nm", "scm20d", "scm1d"]
methods = ["flowgem", "mice", "gain", "hyperimpute", "miri", "bayes", "missdiff", "knewimp"]

n_runs = 10

param_vals = {
    "init": "mice",
    "T": 1000,
    "eta": 0.01
}

In [ ]:
# Define file paths
save_dir = f"{base_path}/results/realdata"
log_file = os.path.join(save_dir, 'parameter_log.csv')
impute_dir = f"{save_dir}/imputations"
metric_dir = f"{save_dir}/metrics"
init_dir = f"{save_dir}/initializations"

In [ ]:
# install R packages if necessary
if 'bayes' in methods:
    os.environ['RENV_CONFIG_AUTOLOAD_ENABLED'] = 'FALSE'
    os.environ['R_PROFILE_USER'] = ''
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
    from rpy2.robjects.conversion import localconverter

    !apt-get install -y r-cran-mcmcpack
    !apt-get install -y r-cran-rfast

    !apt-get install -y r-cran-mcmcpack
    !apt-get install -y r-cran-rfast

    ro.r('''
        packages <- c("mvtnorm", "Rfast")
        for (pkg in packages) {
            if (!requireNamespace(pkg, quietly = TRUE)) {
                install.packages(pkg, repos="https://cloud.r-project.org")
            }
        }
    ''')

    ro.r(f'''
        source("{base_path}/Bayesian/Density_Estimation_Final.R")
        source("{base_path}/Bayesian/MHAlgorithm.R")
        source("{base_path}/Bayesian/helper.R")
    ''')

## Define init function and evaluation metrics

In [ ]:
def impute_bootstrap_per_col(X, M):
    n, d = X.shape
    for j in range(d):
        observed_mask = M[:, j] == 1
        missing_mask = M[:, j] == 0

        observed_values = X[observed_mask, j]
        if observed_values.numel() == 0:
            raise ValueError(f"Value of column {j} is always missing. Need to be observed at least once.")

        num_missing = missing_mask.sum()
        if num_missing > 0:
            rand_idx = torch.randint(0, observed_values.shape[0], (num_missing,), device=X.device)
            X[missing_mask, j] = observed_values[rand_idx]


def energy_distance(X, Y, scale):
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl

    XY = cdist(X, Y)
    XX = cdist(X, X)
    YY = cdist(Y, Y)
    return (2 * XY.mean() - XX.mean() - YY.mean()) * X.shape[0] / 2

def energy_distance_faster(X, Y, scale):
    if torch.max(torch.abs(X)).item() > 1e100 and torch.max(torch.abs(Y)).item() < 1e100:
        return torch.inf
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl

    n = X.shape[0]
    xx_mean = pdist(X).sum() * 2 / (n * n)   # divide by n^2 to match cdist mean
    yy_mean = pdist(Y).sum() * 2 / (n * n)   # same for Y
    xy_mean = cdist(X, Y).mean()

    return (2 * xy_mean - xx_mean - yy_mean) * n / 2

def energy_distance_fixed_X(X, XX_mean, Y):
    XY_mean = cdist(X, Y).mean()
    n = len(Y)
    YY_mean = pdist(Y).mean() * (n - 1) / n
    return 2 * XY_mean - XX_mean - YY_mean


def W2(X, Y, scale):
    if torch.max(torch.abs(X)).item() > 1e100 and torch.max(torch.abs(Y)).item() < 1e100:
        return torch.inf
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl
    else:
        X = np.array(X)
        Y = np.array(Y)

    dists = ((X[:, None] - Y) ** 2).sum(2) / 2.
    n = X.shape[0]
    return ot.emd2(np.ones(n) / n, np.ones(n) / n, dists, numItermax=1_000_000)

def median_heuristic_sigma(X, Y, eps=1e-12):
    X = np.asarray(X)
    Y = np.asarray(Y)

    Z = np.concatenate([X, Y], axis=0)  # (n+m, d)

    # pairwise squared distances
    Z = Z.astype(np.float64, copy=False)
    Z_norm = (Z**2).sum(axis=1, keepdims=True)          # (N,1)
    dist2 = Z_norm + Z_norm.T - 2.0 * (Z @ Z.T)       # (N,N)

    # off-diagonal entries only
    N = dist2.shape[0]
    off_diag = dist2[~np.eye(N, dtype=bool)]

    sigma2 = np.median(off_diag)
    sigma2 = max(sigma2, eps)
    return np.sqrt(sigma2)

def MMD(X, Y, scale, sigma=None):
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl
    else:
        X = np.array(X)
        Y = np.array(Y)

    if sigma is None:
        sigma = median_heuristic_sigma(X, Y)

    sigma2 = sigma ** 2

    def rbf(a, b):
        a2 = (a ** 2).sum(axis=1, keepdims=True)   # (n,1)
        b2 = (b ** 2).sum(axis=1, keepdims=True).T # (1,m)
        dist2 = a2 + b2 - 2.0 * (a @ b.T)          # (n,m)
        return np.exp(-dist2 / (2.0 * sigma2))

    Kxx = rbf(X, X)
    Kyy = rbf(Y, Y)
    Kxy = rbf(X, Y)

    return Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()

## Generate initial particles (used for FLOWGEM, MIRI, MissDiff, and NewImp)

In [ ]:
for dataset in datasets:
    for run in range(n_runs):
        torch.manual_seed(run)
        np.random.seed(run)

        if os.path.exists(f"{init_dir}/init_{dataset}_{param_vals["init"]}_{run}.pt"):
            continue

        Xstar_df = pyreadr.read_r(f"{base_path}/datasets/split_test/test.{dataset}.RDS")[None]
        X_miss_df = pyreadr.read_r(f"{base_path}/datasets/split_amputed/mar.{dataset}.RDS")[None]
        ## M=1-M in the paper
        M_np = (X_miss_df.notna()).astype(int).values
        M_tensor = torch.tensor(M_np, dtype=torch.float64)
        X0_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)

        if param_vals["init"] == "ColBT":
            # bootstrap imputation
            impute_bootstrap_per_col(X0_tensor, M_tensor) # modifies X0 in-place

            # standardize X0 using observed values only
            Xmiss_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)
            col_std = torch.nanmean((Xmiss_tensor - torch.nanmean(Xmiss_tensor, dim=0)) ** 2, dim=0) ** 0.5
            col_mean = torch.nanmean(Xmiss_tensor, dim=0)

        elif param_vals["init"] == "mice":
            # Use mice imputation with 10 iterations as X0
            Xmiss_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)

            col_std = torch.nanmean((Xmiss_tensor - torch.nanmean(Xmiss_tensor, dim=0)) ** 2, dim=0) ** 0.5
            col_mean = torch.nanmean(Xmiss_tensor, dim=0)

            imputer = Imputers().get("mice", max_iter=10, random_state=run)
            print("Imputing with MICE...")
            df = imputer.fit_transform(X_miss_df)
            X0_np = df.values
            X0_tensor = torch.tensor(X0_np, dtype=torch.float64)

            print(np.isnan(X0_np).any())

        else:
            raise NotImplementedError("This initialization technique is not implemented (yet).")

        X0_std = (X0_tensor - col_mean) / col_std

        # Standardize Xstar (to pass to miri method)
        Xstar_tensor = torch.tensor(Xstar_df.values, dtype=torch.float64)
        Xstar_std = (Xstar_tensor - torch.mean(Xstar_tensor, dim=0)) / torch.std(Xstar_tensor, dim=0)

        init_save = {"X0": X0_tensor,
                     "X0_std": X0_std,
                     "Xstar_std": Xstar_std,
                     "col_mean": col_mean,
                     "col_std": col_std}

        torch.save(init_save, f"{init_dir}/init_{dataset}_{param_vals["init"]}_{run}.pt")

## Run imputation methods and compute metrics (only those that are not available yet)

In [ ]:
for dataset in datasets:
    print(f"#=============================#")
    print(f"#========= {dataset} =========#")
    print(f"#=============================#")
    for method in methods:
        print(f"#-----------------------------#")
        print(f"#---------- {method} ---------#")
        print(f"#-----------------------------#")

        needed_runs = [k for k in range(n_runs) if not os.path.exists(f"{metric_dir}/metrics_{dataset}_{method}_{k}.pt")]

        if len(needed_runs) == 0:
            continue

        Xstar_df = pyreadr.read_r(f"{base_path}/datasets/split_test/test.{dataset}.RDS")[None]
        X_miss_df = pyreadr.read_r(f"{base_path}/datasets/split_amputed/mar.{dataset}.RDS")[None]

        for run in needed_runs:
            #============= GENERATION ==============
            if os.path.exists(f"{impute_dir}/imputation_{dataset}_{method}_{run}.pt"):
                imp_save = torch.load(f"{impute_dir}/imputation_{dataset}_{method}_{run}.pt", weights_only=False)
            else:
                torch.manual_seed(run)
                np.random.seed(run)

                ## M=1-M in the paper
                M_np = (X_miss_df.notna()).astype(int).values
                M_tensor = torch.tensor(M_np, dtype=torch.float64)
                X0_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)

                if method in ["miri", "missdiff", "knewimp", "flowgem"]:
                    init_save = torch.load(f"{init_dir}/init_{dataset}_{param_vals["init"]}_{run}.pt", weights_only=False)

                    if method == "flowgem":
                        start = time.time()
                        Xhats = sample_flowgem(init_save["X0_std"], init_save["X0_std"], M_tensor, T=param_vals["T"], eta=param_vals["eta"], grad_tol=0.01)[0]
                        runtime = time.time() - start
                        Xhats = [(torch.tensor(Xhat, dtype=torch.float64) * init_save["col_std"] + init_save["col_mean"]) for Xhat in Xhats]

                    else:
                        start = time.time()
                        Xhat = impute_now(init_save["X0_std"], M_tensor, init_save["Xstar_std"], method, max_rounds=15, batchsize=500, maxepochs=900, odesteps=100)[0] ##Setting of images that fits together with learning rate
                        runtime = time.time() - start
                        Xhats = [torch.tensor(Xhat, dtype=torch.float64) * init_save["col_std"] + init_save["col_mean"]]

                elif method in ["gain", "knn", "hyperimpute", "mice"]:
                    if method == "knn":
                        imputer = KNNImputer(n_neighbors=5)
                    else:
                        imputer = Imputers().get(method, random_state=run)
                    start = time.time()
                    X_imp = imputer.fit_transform(X_miss_df.values)
                    runtime = time.time() - start
                    Xhats = [torch.tensor(np.array(X_imp), dtype=torch.float64)]

                elif method == "bayes":
                    ro.r("set.seed(123)")

                    # Convert pandas -> R
                    X_df = X_miss_df.copy()
                    X_df.columns = [f"x{i}" for i in range(X_df.shape[1])]
                    with localconverter(ro.default_converter + pandas2ri.converter):
                        r_df = ro.conversion.py2rpy(X_df)

                    d = X_df.shape[1]
                    n = X_df.shape[0]

                    dp_gmm = ro.globalenv['dp_gmm_shared_Sigma']
                    sample_pp = ro.globalenv['sample_posterior_predictive']

                    start = time.time()
                    tmp = dp_gmm(
                        X=r_df,
                        mu0=ro.FloatVector([0] * d),
                        tau0_sq=d+1,
                        nu0=d+2,
                        Psi0=ro.r['diag'](d),
                        niter= 800,
                        nburn= 300
                    )
                    r_Xhats = sample_pp(tmp, n_pred=n)
                    runtime = time.time() - start

                    with localconverter(ro.default_converter + pandas2ri.converter):
                        completed_df = ro.conversion.rpy2py(r_Xhats)

                    X_imputed = torch.tensor(completed_df, dtype=torch.float64)
                    Xhats = [X_imputed]

                else:
                    raise NotImplementedError("Method not implemented so far.")

                # Xhats_arr = Xhats[-1].detach().cpu().numpy() if isinstance(Xhats[-1], torch.Tensor) else np.array(Xhats[-1])
                # Xhats_df = pd.DataFrame(Xhats_arr.astype(np.float64))
                Xhat_tensor = Xhats[-1].detach() if isinstance(Xhats[-1], torch.Tensor) else torch.as_tensor(Xhats[-1])

                imp_save = {"Xhat": Xhat_tensor, "runtime": runtime}

                torch.save(imp_save, f"{impute_dir}/imputation_{dataset}_{method}_{run}.pt")

            #============= EVALUATION ==============#
            Xhat = imp_save["Xhat"]
            Xstar_tensor = torch.tensor(Xstar_df.values, dtype=torch.float64)
            metric_save = { "runtime": imp_save["runtime"],
                            "energy_scl": energy_distance_faster(Xstar_tensor, Xhat, scale=True),
                            "W2": W2(Xstar_tensor, Xhat, scale=False),
                            "MMD": MMD(Xstar_tensor, Xhat, scale=False)}

            print(metric_save)
            torch.save(metric_save, f"{metric_dir}/metrics_{dataset}_{method}_{run}.pt")

        #============= AGGREGATION ==============#
        rows = []
        for run in range(n_runs):
            metric_save = torch.load(f"{metric_dir}/metrics_{dataset}_{method}_{run}.pt", weights_only=False)
            rows.append(metric_save)
        df = pd.DataFrame(rows)

        agg = { "runtime_mean": float(df["runtime"].mean()),
                "runtime_std": float(df["runtime"].std(ddof=1)),
                "energy_scl_mean": float(df["energy_scl"].mean()),
                "energy_scl_std": float(df["energy_scl"].std(ddof=1)),
                "W2_mean": float(df["W2"].mean()),
                "W2_std": float(df["W2"].std(ddof=1)),
                "MMD_mean": float(df["MMD"].mean()),
                "MMD_std": float(df["MMD"].std(ddof=1))}

        torch.save(agg, f"{save_dir}/results_{dataset}_{method}.pt")

## Analyze the results

In [ ]:
metrics = ["energy_scl", "W2", "MMD"] #"runtime"

datasets = list(reversed(datasets))

title_dict = {'gain': 'GAIN',
              'hyperimpute': 'Hyperimpute',
              'miri': 'MIRI',
              'bayes': 'Bayes',
              'flowgem': 'FLOWGEM',
              'mice': 'MICE',
              'missdiff': 'MissDiff',
              'knewimp': 'NewImp'}

In [ ]:
def load_agg(dataset: str, method: str):
    path = os.path.join(save_dir, f"results_{dataset}_{method}.pt")
    if os.path.exists(path):
        agg = torch.load(path, map_location="cpu")
    else:
        agg = { "runtime_mean": np.nan,
                "runtime_std": np.nan,
                "energy_scl_mean": np.nan,
                "energy_scl_std": np.nan,
                "W2_mean": np.nan,
                "W2_std": np.nan,
                "MMD_mean": np.nan,
                "MMD_std": np.nan}
    return agg


def format_cell(agg, metric: str):
    mean = agg[f"{metric}_mean"]
    std = agg[f"{metric}_std"]
    return f"{mean:.3f} ({std:.2f})"


def best_two_indices(values, minimize: bool):
    values = np.asarray(values, dtype=float)
    # best = smallest if minimize else largest
    order = np.argsort(values)          # ascending
    if not minimize:
        order = order[::-1]             # descending
    return int(order[0]), int(order[1])


def latex_table(metric: str, datasets, methods, relative=True):
    minimize = True
    lines = []
    lines += [
        # r"\begin{table}[t]",
        # r"\centering",
        # r"\small",
        rf"\caption{{{metric} mean (std)}}",
        rf"\begin{{tabular}}{{ll{'r'*len(methods)}}}",
        r"\toprule",
        " & ".join([r"\textbf{(a)}", "Dataset", *[title_dict[m] for m in methods]]) + r" \\",
        r"\midrule",
    ]

    for d in datasets:
        aggs = [load_agg(d, m) for m in methods]
        means = [a[f"{metric}_mean"] for a in aggs]
        best_i, second_i = best_two_indices(means, minimize=minimize)

        row = [" ", d]
        for j, (mth, agg) in enumerate(zip(methods, aggs)):
            mean = agg[f"{metric}_mean"]
            std = agg[f"{metric}_std"]

            if np.isnan(mean):
                row.append("--")
                continue

            if relative:
                mean = mean / aggs[best_i][f"{metric}_mean"]
                std = std / aggs[best_i][f"{metric}_mean"]

            if mean > 100:
                if relative:
                    row.append(f"$>100$")
                    continue
                elif mean > 1e3:
                    row.append(f"$>10^3$")
                    continue

            mean_str = f"{mean:.2f}" if relative else f"{mean:.3f}"

            if j == best_i:
                mean_str = rf"\textbf{{{mean_str}}}"
            elif j == second_i:
                mean_str = rf"\textbf{{{mean_str}}}" if relative else rf"\underline{{{mean_str}}}"

            std_str = f"{std:.2f}" if not np.isnan(std) else "  ----  "
            row.append(f"{mean_str} ({std_str})")

        lines.append(" & ".join(row) + r" \\")

    lines += [r"\bottomrule", r"\end{tabular}"] #, r"\end{table}"]
    return "\n".join(lines)


# Generate tables
for metric in metrics:
    tex = latex_table(metric, datasets, methods)
    print()
    print(tex)